## 1. Installation

In [1]:
!pip install -q -U "transformers>=4.51" accelerate "bitsandbytes>=0.46.1" peft
print("✅ Installation terminée. Maintenant : Exécution → Redémarrer la session")

✅ Installation terminée. Maintenant : Exécution → Redémarrer la session


## 1b. Vérification

In [2]:
import torch, transformers, bitsandbytes, peft
print("transformers :", transformers.__version__, "| bitsandbytes :", bitsandbytes.__version__, "| GPU :", torch.cuda.is_available())
assert torch.cuda.is_available(), "⚠️ Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4"

transformers : 5.24.0 | bitsandbytes : 0.50.2 | GPU : True


## 2. Connexion à Hugging Face

In [3]:
from huggingface_hub import login
try:
    login()
except Exception as e:
    print("Connexion ignorée :", e)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


## 3. Le corpus de phrases


In [5]:
import os, glob, math, shutil
import pandas as pd
from google.colab import drive, files

drive.mount("/content/drive")
DOSSIER = "/content/drive/MyDrive/SmartCampus_LLM"
os.makedirs(DOSSIER, exist_ok=True)

CORPUS = os.path.join(DOSSIER, "corpus_perplexite.csv")
if not os.path.exists(CORPUS):
    envoye = files.upload()                      # choisissez corpus_perplexite.csv
    nom = list(envoye.keys())[0]
    with open(CORPUS, "wb") as f:
        f.write(envoye[nom])
    print("Copie enregistrée dans le Drive :", CORPUS)
else:
    print("Corpus trouvé dans le Drive :", CORPUS, "(pour le changer, remplacez ce fichier dans le Drive)")

corpus = pd.read_csv(CORPUS, dtype=str).fillna("")
corpus = corpus[corpus["phrase"].str.strip() != ""].reset_index(drop=True)
corpus["phrase_id"] = corpus.index.astype(str)
print(corpus["langue"].value_counts())

# Vérification sécurisée de la colonne 'source'
if "source" in corpus.columns:
    if (corpus["source"].str.contains("exemple", case=False)).any():
        print("⚠️ Le corpus contient encore des phrases d'exemple : remplacez-les par de vraies phrases avant de présenter les résultats.")
else:
    print("Information : Pas de colonne 'source' détectée dans le corpus.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Corpus trouvé dans le Drive : /content/drive/MyDrive/SmartCampus_LLM/corpus_perplexite.csv (pour le changer, remplacez ce fichier dans le Drive)
langue
Tounsi      50
Arabizi     50
Français    50
Anglais     50
Name: count, dtype: int64
Information : Pas de colonne 'source' détectée dans le corpus.


## 4. Les modèles

In [6]:
MODELES = [
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen3-8B",
    "Artess99/derja-qwen3-8b-derja-v2",       # adaptateur LoRA posé sur Qwen3-8B (chargé avec peft)
    # "humain-ai/ALLaM-7B-Instruct-preview",  # spécialisé arabe
    # "CohereLabs/aya-expanse-8b",            # accepter les conditions sur Hugging Face + HF_TOKEN
]

## 5. Fonctions
Pour chaque phrase, on calcule la **log-vraisemblance négative** (NLL) de tous ses tokens.
On ajoute un retour à la ligne devant la phrase pour que le **premier mot soit aussi évalué**.

In [7]:
import gc, shutil, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

def charger(nom):
    tok = AutoTokenizer.from_pretrained(nom)
    # Un adaptateur LoRA (ex. Artess99) est posé automatiquement sur son modèle de base grâce à peft.
    mod = AutoModelForCausalLM.from_pretrained(nom, quantization_config=bnb, device_map="auto")
    mod.eval()
    return tok, mod

@torch.no_grad()
def nll_phrase(tok, mod, phrase):
    prefixe = tok("\n", add_special_tokens=False)["input_ids"]
    ids = tok(phrase, add_special_tokens=False)["input_ids"]
    x = torch.tensor([prefixe + ids], device=mod.device)
    labels = torch.tensor([[-100] * len(prefixe) + ids], device=mod.device)   # on n'évalue que la phrase
    perte = mod(input_ids=x, labels=labels).loss.item()                     # NLL moyenne par token (nats)
    return perte * len(ids), len(ids)                                        # NLL totale, nombre de tokens

def liberer(mod):
    del mod; gc.collect(); torch.cuda.empty_cache()

## 6. Calcul

In [8]:
FICHIER = os.path.join(DOSSIER, "perplexite_details.csv")
details = pd.read_csv(FICHIER, dtype={"phrase_id": str}).to_dict("records") if os.path.exists(FICHIER) else []
print(len(details), "phrases déjà calculées")

for nom in MODELES:
    faits = {r["phrase_id"] for r in details if r["modele"] == nom}
    a_faire = corpus[~corpus["phrase_id"].isin(faits)]
    if a_faire.empty:
        print("✔ déjà terminé :", nom); continue
    print("\n" + "=" * 60 + f"\n{nom} : {len(a_faire)} phrases")
    try:
        tok, mod = charger(nom)
    except Exception as e:
        print("❌ Impossible de charger :", e); continue
    for _, r in a_faire.iterrows():
        nll, n_tok = nll_phrase(tok, mod, r["phrase"])
        details.append({"modele": nom, "phrase_id": r["phrase_id"], "langue": r["langue"], "nll": nll,
                        "tokens": n_tok, "caracteres": len(r["phrase"]), "mots": len(r["phrase"].split())})
        if len(details) % 20 == 0:
            pd.DataFrame(details).to_csv(FICHIER, index=False)   # sauvegarde régulière dans le Drive
    pd.DataFrame(details).to_csv(FICHIER, index=False)
    liberer(mod)
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)
print("✅ Terminé")

0 phrases déjà calculées

Qwen/Qwen2.5-7B-Instruct : 200 phrases


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (29 outputs left out)

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 87.4MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

✅ Terminé


## 7. Résultats
- **BPC** (bits par caractère) : la mesure principale, **comparable entre tous les modèles**. Plus bas = mieux.
- Comparez aussi les langues **pour un même modèle** : un BPC bien plus haut en Arabizi qu'en français montre que le modèle connaît mal l'Arabizi.

In [9]:
df = pd.read_csv(FICHIER)
g = df.groupby(["modele", "langue"])[["nll", "tokens", "caracteres", "mots"]].sum()
g["BPC"] = g["nll"] / math.log(2) / g["caracteres"]
g["perplexite_par_mot"] = (g["nll"] / g["mots"]).apply(math.exp)
g["perplexite_par_token"] = (g["nll"] / g["tokens"]).apply(math.exp)
g["phrases"] = df.groupby(["modele", "langue"]).size()
resultats = g.reset_index()

bpc = resultats.pivot(index="modele", columns="langue", values="BPC").round(3)
ppl_mot = resultats.pivot(index="modele", columns="langue", values="perplexite_par_mot").round(1)
print("Bits par caractère (plus bas = mieux)"); display(bpc)
print("Perplexité par mot (plus bas = mieux)"); display(ppl_mot)

Bits par caractère (plus bas = mieux)


langue,Anglais,Arabizi,Français,Tounsi
modele,,,,
Artess99/derja-qwen3-8b-derja-v2,1.557,4.161,1.445,3.404
Qwen/Qwen2.5-7B-Instruct,1.372,4.353,1.275,3.352
Qwen/Qwen3-8B,1.393,4.339,1.367,3.558


Perplexité par mot (plus bas = mieux)


langue,Anglais,Arabizi,Français,Tounsi
modele,,,,
Artess99/derja-qwen3-8b-derja-v2,344.9,18228292.2,402.5,212540.1
Qwen/Qwen2.5-7B-Instruct,172.2,39465024.2,198.7,176343.0
Qwen/Qwen3-8B,186.4,37374243.8,290.7,370571.3


## 8. Export Excel

In [10]:
XLSX = os.path.join(DOSSIER, "perplexite_resultats.xlsx")
with pd.ExcelWriter(XLSX, engine="openpyxl") as w:
    bpc.to_excel(w, sheet_name="BPC")
    ppl_mot.to_excel(w, sheet_name="perplexite_par_mot")
    resultats.to_excel(w, sheet_name="detail_par_langue", index=False)
    df.to_excel(w, sheet_name="detail_par_phrase", index=False)
files.download(XLSX)
print("✅ Excel enregistré dans le Drive et téléchargé :", XLSX)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Excel enregistré dans le Drive et téléchargé : /content/drive/MyDrive/SmartCampus_LLM/perplexite_resultats.xlsx
